In [2]:
!pip install -q transformers datasets pillow scikit-learn loguru tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 906.6 kB/s eta 0:00:000:00:01


In [3]:
import os

# Найдём где лежит архив
data_path = '/kaggle/input/datasets/lh7ng0cifjog1/metrical-books-ocr/'
print("Содержимое директории:")
for f in os.listdir(data_path):
    print(f"  {f}")

# Проверим все файлы рекурсивно
print("\nВсе файлы:")
for root, dirs, files in os.walk(data_path):
    for f in files[:5]:
        print(f"  {os.path.join(root, f)}")

Содержимое директории:
  finetune_data_preprocessed
  finetune_jsonl

Все файлы:
  /kaggle/input/datasets/lh7ng0cifjog1/metrical-books-ocr/finetune_data_preprocessed/._finetune_data_preprocessed
  /kaggle/input/datasets/lh7ng0cifjog1/metrical-books-ocr/finetune_data_preprocessed/finetune_data_preprocessed/6234a5470343ee4f1da10452_86_31_preprocessed.png
  /kaggle/input/datasets/lh7ng0cifjog1/metrical-books-ocr/finetune_data_preprocessed/finetune_data_preprocessed/._62349ed30343ee4f1da102da_226_30_preprocessed.png
  /kaggle/input/datasets/lh7ng0cifjog1/metrical-books-ocr/finetune_data_preprocessed/finetune_data_preprocessed/._6254180fbb49f011c7aaf27e_138_11_preprocessed.png
  /kaggle/input/datasets/lh7ng0cifjog1/metrical-books-ocr/finetune_data_preprocessed/finetune_data_preprocessed/6253aba20923d1ec88f25004_91_6_preprocessed.png
  /kaggle/input/datasets/lh7ng0cifjog1/metrical-books-ocr/finetune_data_preprocessed/finetune_data_preprocessed/62526eab0923d1ec88f1f083_852_12_preprocessed.png

In [4]:
import json, random
from pathlib import Path

# Загружаем JSONL файлы (пути уже ведут к preprocessed!)
jsonl_src = Path('/kaggle/input/datasets/lh7ng0cifjog1/metrical-books-ocr/finetune_jsonl/finetune_data')

dataset = []
for split in ['train.jsonl', 'val.jsonl']:
    with open(jsonl_src / split, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                dataset.append(json.loads(line))

print(f"Загружено записей: {len(dataset)}")

# Пути в JSONL уже ведут на Mac: /Users/mnijonshuti/...
# Нужно заменить на /kaggle/working/
import os
for item in dataset:
    # Заменяем путь на Kaggle
    item['image_path'] = item['image_path'].replace(
        '/Users/mnijonshuti/smart-match/finetune_data_preprocessed/',
        '/kaggle/working/finetune_data_preprocessed/'
    )

# Проверяем, что первые 5 файлов существуют
print("\nПроверка путей:")
for i in range(min(5, len(dataset))):
    exists = os.path.exists(dataset[i]['image_path'])
    print(f"  [{i}] {exists} — {dataset[i]['text'][:50]}")

# Split
random.shuffle(dataset)
n_val = max(1, len(dataset) // 10)
val_data = dataset[:n_val]
train_data = dataset[n_val:]
print(f"\nTrain: {len(train_data)}, Val: {len(val_data)}")

Загружено записей: 4500

Проверка путей:
  [0] False — Протоiерей Николай Ливановъ
  [1] False — славнаго исповъданiя, -2
  [2] False — Наровчатскiй Государ- нинъ Ксенофонтъ
  [3] False — Ефимъ Парфе -
  [4] False — равовъ иствъдывалъ и прiоб-

Train: 4050, Val: 450


In [5]:
import torch, gc, os

# Очищаем всю память
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Проверяем свободную память
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"Свободно: {free/1e9:.1f} GB / {total/1e9:.1f} GB")
    
# Free up any lingering allocations
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print("Память очищена!")

Свободно: 15.5 GB / 15.6 GB
Память очищена!


In [6]:
import json, random
from pathlib import Path

# Загружаем JSONL
jsonl_src = Path('/kaggle/input/datasets/lh7ng0cifjog1/metrical-books-ocr/finetune_jsonl/finetune_data')

dataset = []
for split in ['train.jsonl', 'val.jsonl']:
    with open(jsonl_src / split, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                dataset.append(json.loads(line))

# Заменяем пути
for item in dataset:
    item['image_path'] = item['image_path'].replace(
        '/Users/mnijonshuti/smart-match/finetune_data_preprocessed/',
        '/kaggle/working/finetune_data_preprocessed/'
    )

# Копируем preprocessed если нужно
import shutil
preproc_dst = Path('/kaggle/working/finetune_data_preprocessed')
if not preproc_dst.exists():
    preproc_src = Path('/kaggle/input/datasets/lh7ng0cifjog1/metrical-books-ocr/finetune_data_preprocessed')
    inner = list(preproc_src.glob('*'))
    for item in inner:
        if item.is_dir() and 'finetune_data_preprocessed' in item.name:
            shutil.copytree(item, preproc_dst)
            break

# Split
random.shuffle(dataset)
n_val = max(1, len(dataset) // 10)
val_data = dataset[:n_val]
train_data = dataset[n_val:]
print(f"Train: {len(train_data)}, Val: {len(val_data)}")
print(f"Пример: {train_data[0]['text'][:50]}")

Train: 4050, Val: 450
Пример: Михаилъ.


In [7]:
import torch, gc
from loguru import logger
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, get_scheduler
from pathlib import Path
import torchvision.transforms as T

OUTPUT_DIR = Path("/kaggle/working/trocr-finetuned1")
MODEL_NAME = "microsoft/trocr-base-printed"
BATCH_SIZE = 32
EPOCHS = 10
LR = 2e-5
LIMIT = 0  # Все данные!

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Device: {device}")

gc.collect()
torch.cuda.empty_cache()

# Data augmentation
train_transform = T.Compose([
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.RandomAffine(degrees=2, translate=(0.05, 0.05)),
])

class OcrData(Dataset):
    def __init__(self, items, processor, max_len=128, augment=False):
        self.items = items
        self.processor = processor
        self.max_len = max_len
        self.augment = augment
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        item = self.items[idx]
        try:
            img = Image.open(item["image_path"]).convert("RGB")
            if self.augment:
                img = train_transform(img)
        except:
            img = Image.new("RGB", (384, 64), 255)
        pix = self.processor(img, return_tensors="pt").pixel_values[0]
        lbl = self.processor.tokenizer(
            item['text'], padding='max_length', max_length=self.max_len,
            truncation=True, return_tensors='pt',
        ).input_ids[0]
        return {'pixel_values': pix, 'labels': lbl}

def collate(batch):
    pix = torch.stack([b['pixel_values'] for b in batch])
    lbl = torch.stack([b['labels'] for b in batch])
    lbl[lbl == processor.tokenizer.pad_token_id] = -100
    return {'pixel_values': pix.to(device), 'labels': lbl.to(device)}

processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME).to(device)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.generation_config.decoder_start_token_id = model.config.decoder_start_token_id
model.generation_config.pad_token_id = model.config.pad_token_id

# Freeze encoder (иначе OOM)
for p in model.encoder.parameters():
    p.requires_grad = False

decoder_params = sum(p.numel() for p in model.decoder.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
logger.info(f"Decoder fine-tune: {decoder_params/1e6:.1f}M / {total_params/1e6:.1f}M params")

# Все данные
train_subset = train_data
val_subset = val_data
logger.info(f"Train: {len(train_subset)}, Val: {len(val_subset)}")

train_loader = DataLoader(
    OcrData(train_subset, processor, augment=True), batch_size=BATCH_SIZE,
    shuffle=True, collate_fn=collate, num_workers=0
)
val_loader = DataLoader(
    OcrData(val_subset, processor, augment=False), batch_size=BATCH_SIZE,
    shuffle=False, collate_fn=collate, num_workers=0
)

opt = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, weight_decay=0.01
)

total_steps = EPOCHS * len(train_loader)
sched = get_scheduler('cosine', opt, int(0.1 * total_steps), total_steps)

logger.info(f"Training: {EPOCHS} epochs, {total_steps} steps (all {len(train_subset)} data)")

best_val = float('inf')
OUTPUT_DIR.mkdir(exist_ok=True)

for epoch in range(EPOCHS):
    model.train()
    tloss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    for batch in pbar:
        loss = model(**batch).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()
        opt.zero_grad()
        tloss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    logger.info(f'Epoch {epoch+1}: train_loss={tloss/len(train_loader):.4f}')

    model.eval()
    vloss = 0.0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc='Val'):
            loss = model(**batch).loss
            vloss += loss.item()
    avg_vloss = vloss / len(val_loader)
    logger.info(f'Epoch {epoch+1}: val_loss={avg_vloss:.4f}')

    if avg_vloss < best_val:
        best_val = avg_vloss
        model.save_pretrained(str(OUTPUT_DIR))
        processor.save_pretrained(str(OUTPUT_DIR))
        logger.info(f'Saved (val_loss={avg_vloss:.4f})')

# Финальный тест
logger.info("Testing:")
model.eval()
for i in range(min(10, len(val_subset))):
    item = val_subset[i]
    img = Image.open(item['image_path']).convert('RGB')
    pix = processor(img, return_tensors='pt').pixel_values.to(device)
    with torch.no_grad():
        gen = model.generate(pix, max_length=64, num_beams=4, early_stopping=True)
    pred = processor.batch_decode(gen, skip_special_tokens=True)[0]
    match = 'MATCH' if pred.strip() == item['text'].strip() else 'OTHERWISE'
    logger.info(f'  {match} [{i+1}] GT=«{item["text"]}»')
    logger.info(f'       Pred=«{pred}»')

logger.info("Done!")

2026-07-30 05:42:22.476 | INFO     | __main__:<cell line: 0>:18 - Device: cuda


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

2026-07-30 06:42:55.828 | INFO     | __main__:<cell line: 0>:129 - Saved (val_loss=0.8137)
Epoch 8/10: 100%|██████████| 127/127 [08:02<00:00,  3.80s/it, loss=0.8502]
2026-07-30 06:50:58.637 | INFO     | __main__:<cell line: 0>:114 - Epoch 8: train_loss=0.7728
Val: 100%|██████████| 15/15 [00:32<00:00,  2.15s/it]
2026-07-30 06:51:30.947 | INFO     | __main__:<cell line: 0>:123 - Epoch 8: val_loss=0.7828


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-30 06:51:34.313 | INFO     | __main__:<cell line: 0>:129 - Saved (val_loss=0.7828)
Epoch 9/10: 100%|██████████| 127/127 [08:02<00:00,  3.80s/it, loss=1.1247]
2026-07-30 06:59:36.628 | INFO     | __main__:<cell line: 0>:114 - Epoch 9: train_loss=0.7431
Val: 100%|██████████| 15/15 [00:32<00:00,  2.15s/it]
2026-07-30 07:00:08.952 | INFO     | __main__:<cell line: 0>:123 - Epoch 9: val_loss=0.7753


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-30 07:00:12.338 | INFO     | __main__:<cell line: 0>:129 - Saved (val_loss=0.7753)
Epoch 10/10: 100%|██████████| 127/127 [08:02<00:00,  3.80s/it, loss=0.7126]
2026-07-30 07:08:14.621 | INFO     | __main__:<cell line: 0>:114 - Epoch 10: train_loss=0.7289
Val: 100%|██████████| 15/15 [00:32<00:00,  2.15s/it]
2026-07-30 07:08:46.857 | INFO     | __main__:<cell line: 0>:123 - Epoch 10: val_loss=0.7753
2026-07-30 07:08:46.858 | INFO     | __main__:<cell line: 0>:132 - Testing:
2026-07-30 07:08:48.590 | INFO     | __main__:<cell line: 0>:142 -   OTHERWISE [1] GT=«Николай Беримишевскiй»
2026-07-30 07:08:48.591 | INFO     | __main__:<cell line: 0>:143 -        Pred=«Алексъй Берониновскiй»
2026-07-30 07:08:48.946 | INFO     | __main__:<cell line: 0>:142 -   OTHERWISE [2] GT=«комъ»
2026-07-30 07:08:48.947 | INFO     | __main__:<cell line: 0>:143 -        Pred=«каго»
2026-07-30 07:08:49.749 | INFO     | __main__:<cell line: 0>:142 -   OTHERWISE [3] GT=«и: шесть /6/ человек.»
2026-07-30 07:

In [7]:
import torch, gc
from loguru import logger
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, get_scheduler
from pathlib import Path
import torchvision.transforms as T

OUTPUT_DIR = Path("/kaggle/working/trocr-finetuned")
MODEL_NAME = "microsoft/trocr-base-printed"
BATCH_SIZE = 32
EPOCHS = 10              # ← Увеличили до 10
LR = 2e-5
LIMIT = 0                 # ← 0 = все данные!

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Device: {device}")

gc.collect()
torch.cuda.empty_cache()

train_transform = T.Compose([
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.RandomAffine(degrees=2, translate=(0.05, 0.05)),
])

class OcrData(Dataset):
    def __init__(self, items, processor, max_len=128, augment=False):
        self.items = items
        self.processor = processor
        self.max_len = max_len
        self.augment = augment
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        item = self.items[idx]
        try:
            img = Image.open(item["image_path"]).convert("RGB")
            if self.augment:
                img = train_transform(img)
        except:
            img = Image.new("RGB", (384, 64), 255)
        pix = self.processor(img, return_tensors="pt").pixel_values[0]
        lbl = self.processor.tokenizer(
            item['text'], padding='max_length', max_length=self.max_len,
            truncation=True, return_tensors='pt',
        ).input_ids[0]
        return {'pixel_values': pix, 'labels': lbl}

def collate(batch):
    pix = torch.stack([b['pixel_values'] for b in batch])
    lbl = torch.stack([b['labels'] for b in batch])
    lbl[lbl == processor.tokenizer.pad_token_id] = -100
    return {'pixel_values': pix.to(device), 'labels': lbl.to(device)}

processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME).to(device)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.generation_config.decoder_start_token_id = model.config.decoder_start_token_id
model.generation_config.pad_token_id = model.config.pad_token_id

for p in model.encoder.parameters():
    p.requires_grad = False

# Все данные!
train_subset = train_data if LIMIT == 0 else train_data[:LIMIT]
val_subset = val_data if LIMIT == 0 else val_data[:LIMIT//10]

logger.info(f"Train: {len(train_subset)}, Val: {len(val_subset)}")

train_loader = DataLoader(
    OcrData(train_subset, processor, augment=True), batch_size=BATCH_SIZE,
    shuffle=True, collate_fn=collate, num_workers=0
)
val_loader = DataLoader(
    OcrData(val_subset, processor, augment=False), batch_size=BATCH_SIZE,
    shuffle=False, collate_fn=collate, num_workers=0
)

opt = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, weight_decay=0.01
)

total_steps = EPOCHS * len(train_loader)
sched = get_scheduler('cosine', opt, int(0.1 * total_steps), total_steps)

logger.info(f"Training: {EPOCHS} epochs, {total_steps} steps (all {len(train_subset)} data)")

best_val = float('inf')
OUTPUT_DIR.mkdir(exist_ok=True)

for epoch in range(EPOCHS):
    model.train()
    tloss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    for batch in pbar:
        loss = model(**batch).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()
        opt.zero_grad()
        tloss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    logger.info(f'Train loss: {tloss/len(train_loader):.4f}')

    model.eval()
    vloss = 0.0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc='Val'):
            loss = model(**batch).loss
            vloss += loss.item()
    avg_vloss = vloss / len(val_loader)
    logger.info(f'Val loss: {avg_vloss:.4f}')

    if avg_vloss > best_val and epoch > 1:
        logger.warning(f"Val loss вырос! Остановка на эпохе {epoch+1}")
        break

    if avg_vloss < best_val:
        best_val = avg_vloss
        model.save_pretrained(str(OUTPUT_DIR))
        processor.save_pretrained(str(OUTPUT_DIR))
        logger.info(f'Saved (val_loss={avg_vloss:.4f})')

# Тест
logger.info("Testing:")
model.eval()
for i in range(min(10, len(val_subset))):
    item = val_subset[i]
    img = Image.open(item['image_path']).convert('RGB')
    pix = processor(img, return_tensors='pt').pixel_values.to(device)
    with torch.no_grad():
        gen = model.generate(pix, max_length=64, num_beams=4, early_stopping=True)
    pred = processor.batch_decode(gen, skip_special_tokens=True)[0]
    match = '✅' if pred.strip() == item['text'].strip() else '🔄'
    logger.info(f'  {match} [{i+1}] GT=«{item["text"]}» → Pred=«{pred}»')

logger.info("Done!")

2026-07-30 10:50:46.064 | INFO     | __main__:<cell line: 0>:18 - Device: cuda


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

2026-07-30 12:00:44.957 | INFO     | __main__:<cell line: 0>:128 - Saved (val_loss=0.9278)
Epoch 9/10: 100%|██████████| 127/127 [08:07<00:00,  3.84s/it, loss=0.9956]
2026-07-30 12:08:52.745 | INFO     | __main__:<cell line: 0>:109 - Train loss: 0.7555
Val: 100%|██████████| 15/15 [00:32<00:00,  2.17s/it]
2026-07-30 12:09:25.262 | INFO     | __main__:<cell line: 0>:118 - Val loss: 0.9157


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-30 12:09:28.607 | INFO     | __main__:<cell line: 0>:128 - Saved (val_loss=0.9157)
Epoch 10/10: 100%|██████████| 127/127 [08:07<00:00,  3.84s/it, loss=0.8127]
2026-07-30 12:17:35.806 | INFO     | __main__:<cell line: 0>:109 - Train loss: 0.7416
Val: 100%|██████████| 15/15 [00:32<00:00,  2.18s/it]
2026-07-30 12:18:08.527 | INFO     | __main__:<cell line: 0>:118 - Val loss: 0.9156


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-30 12:18:11.931 | INFO     | __main__:<cell line: 0>:128 - Saved (val_loss=0.9156)
2026-07-30 12:18:11.932 | INFO     | __main__:<cell line: 0>:131 - Testing:
2026-07-30 12:18:14.013 | INFO     | __main__:<cell line: 0>:141 -   🔄 [1] GT=«Псоломщикъ Н. Богоявленскiй» → Pred=«Псаломщикъ Н. Богоявленскiй»
2026-07-30 12:18:15.484 | INFO     | __main__:<cell line: 0>:141 -   🔄 [2] GT=«Ефремов, и прибыти» → Pred=«Священникъ Феодорътовъ»
2026-07-30 12:18:17.277 | INFO     | __main__:<cell line: 0>:141 -   🔄 [3] GT=«Священникъ Iосифъ Ефремов» → Pred=«Священникъ Iосифъ Ефремовъ»
2026-07-30 12:18:17.685 | INFO     | __main__:<cell line: 0>:141 -   🔄 [4] GT=«шихся» → Pred=«Тихся»
2026-07-30 12:18:20.259 | INFO     | __main__:<cell line: 0>:141 -   🔄 [5] GT=«Государ крестьянинъ Николай Федоровъ» → Pred=«Государ. крестьянинъ Николай Феодоровъ»
2026-07-30 12:18:22.626 | INFO     | __main__:<cell line: 0>:141 -   🔄 [6] GT=«Кочелаева Феодоръ Михаиловъ Баландинъ» → Pred=«крескова Феодоръ Михаил

In [9]:
# Создайте файл model.zip вручную
!zip -r /kaggle/working/trocr-finetuned.zip /kaggle/working/trocr-finetuned/

updating: kaggle/working/trocr-finetuned/ (stored 0%)
updating: kaggle/working/trocr-finetuned/model.safetensors (deflated 16%)
updating: kaggle/working/trocr-finetuned/processor_config.json (deflated 52%)
updating: kaggle/working/trocr-finetuned/tokenizer_config.json (deflated 50%)
updating: kaggle/working/trocr-finetuned/tokenizer.json (deflated 82%)
updating: kaggle/working/trocr-finetuned/config.json (deflated 74%)
updating: kaggle/working/trocr-finetuned/generation_config.json (deflated 36%)


In [14]:
import base64, os

# Читаем zip файл
with open('/kaggle/working/trocr-finetuned.zip', 'rb') as f:
    data = f.read()

# Кодируем в base64
encoded = base64.b64encode(data).decode()

# Разбиваем на части для удобства
chunk_size = 50000
chunks = [encoded[i:i+chunk_size] for i in range(0, len(encoded), chunk_size)]

print(f"Всего частей: {len(chunks)}")
print(f"Размер: {len(data)/1024/1024:.1f} MB")
print()
print("Скопируйте и выполните на Mac:")

Всего частей: 29878
Размер: 1068.5 MB

Скопируйте и выполните на Mac:


In [ ]:
import torch, gc
from loguru import logger
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, get_scheduler
from pathlib import Path
import torchvision.transforms as T

OUTPUT_DIR = Path("/kaggle/working/trocr-finetuned")
MODEL_NAME = "microsoft/trocr-base-printed"
BATCH_SIZE = 32
EPOCHS = 5              # Меньше эпох!
LR = 2e-5               # Меньше LR!

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Device: {device}")

gc.collect()
torch.cuda.empty_cache()

# Data augmentation (регуляризация!)
train_transform = T.Compose([
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.RandomAffine(degrees=2, translate=(0.05, 0.05)),
])

class OcrData(Dataset):
    def __init__(self, items, processor, max_len=128, augment=False):
        self.items = items
        self.processor = processor
        self.max_len = max_len
        self.augment = augment
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        item = self.items[idx]
        try:
            img = Image.open(item["image_path"]).convert("RGB")
            if self.augment:
                img = train_transform(img)  # Аугментация!
        except:
            img = Image.new("RGB", (384, 64), 255)
        pix = self.processor(img, return_tensors="pt").pixel_values[0]
        lbl = self.processor.tokenizer(
            item['text'], padding='max_length', max_length=self.max_len,
            truncation=True, return_tensors='pt',
        ).input_ids[0]
        return {'pixel_values': pix, 'labels': lbl}

def collate(batch):
    pix = torch.stack([b['pixel_values'] for b in batch])
    lbl = torch.stack([b['labels'] for b in batch])
    lbl[lbl == processor.tokenizer.pad_token_id] = -100
    return {'pixel_values': pix.to(device), 'labels': lbl.to(device)}

processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME).to(device)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.generation_config.decoder_start_token_id = model.config.decoder_start_token_id
model.generation_config.pad_token_id = model.config.pad_token_id

# Freeze encoder
for p in model.encoder.parameters():
    p.requires_grad = False

LIMIT = 10000
train_subset = train_data[:LIMIT]
val_subset = val_data[:LIMIT//10]

# Аугментация только для train
train_loader = DataLoader(
    OcrData(train_subset, processor, augment=True), batch_size=BATCH_SIZE,
    shuffle=True, collate_fn=collate, num_workers=0
)
val_loader = DataLoader(
    OcrData(val_subset, processor, augment=False), batch_size=BATCH_SIZE,
    shuffle=False, collate_fn=collate, num_workers=0
)

opt = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, weight_decay=0.01
)

# Cosine scheduler with warmup
total_steps = EPOCHS * len(train_loader)
sched = get_scheduler('cosine', opt, int(0.1 * total_steps), total_steps)

logger.info(f"Training: {EPOCHS} epochs, {total_steps} steps (lr={LR})")

best_val = float('inf')
OUTPUT_DIR.mkdir(exist_ok=True)

for epoch in range(EPOCHS):
    model.train()
    tloss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    for batch in pbar:
        loss = model(**batch).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()
        opt.zero_grad()
        tloss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    logger.info(f'Train loss: {tloss/len(train_loader):.4f}')

    model.eval()
    vloss = 0.0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc='Val'):
            loss = model(**batch).loss
            vloss += loss.item()
    avg_vloss = vloss / len(val_loader)
    logger.info(f'Val loss: {avg_vloss:.4f}')
    
    # Ранняя остановка при переобучении
    if avg_vloss > best_val and epoch > 1:
        logger.warning(f"Val loss вырос! Остановка.")
        break

    if avg_vloss < best_val:
        best_val = avg_vloss
        model.save_pretrained(str(OUTPUT_DIR))
        processor.save_pretrained(str(OUTPUT_DIR))
        logger.info(f'Saved (val_loss={avg_vloss:.4f})')

# Тест
logger.info("Testing:")
model.eval()
for i in range(min(5, len(val_subset))):
    item = val_subset[i]
    img = Image.open(item['image_path']).convert('RGB')
    pix = processor(img, return_tensors='pt').pixel_values.to(device)
    with torch.no_grad():
        gen = model.generate(pix, max_length=64, num_beams=4, early_stopping=True)
    pred = processor.batch_decode(gen, skip_special_tokens=True)[0]
    logger.info(f'  [{i+1}] GT=«{item["text"]}» → Pred=«{pre you are combod}»')

logger.info("Done!")

In [ ]:
import kagglehub
from pathlib import Path

# Create a new dataset version
kagglehub.dataset_upload(
    handle="lh7ng0cifjog1/trocr-finetuned-model",
    local_path="/kaggle/working/trocr-finetuned1",
    version_notes="Fine-tuned TrOCR on Russian metrical books"
)